# 5.5 导出到运行时：动态形状与数值一致性

> 预估学习时间：45 分钟。资料检索：2026-11-17。
> 配套：[端侧运行时与交付更新](../runtime_update_2026-11.md)。

使用随机初始化的小模型，验证 `torch.onnx.export(dynamo=True)` → ONNX 检查 → ONNX Runtime CPU 执行 → 数值回归 → 输入契约 → 交付元信息。

需要 PyTorch 2.6+、onnx、onnxscript、onnxruntime、numpy（见课程 requirements）。无需下载模型，无需 GPU 或 NPU。CPU 验证不能证明目标设备的委托覆盖、加速比、功耗或真实 LLM 质量。


In [ ]:
import hashlib
import json
import platform
import statistics
import tempfile
from pathlib import Path
from time import perf_counter

import numpy as np
import onnx
import onnxruntime as ort
import onnxscript
import torch

if tuple(int(part) for part in torch.__version__.split("+")[0].split(".")[:2]) < (2, 6):
    raise RuntimeError("Use PyTorch >= 2.6 for the torch.export-based ONNX path.")
torch.manual_seed(42)
torch.set_num_threads(1)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__, "ONNX:", onnx.__version__)
print("ONNX Script:", onnxscript.__version__, "ONNX Runtime:", ort.__version__)
print("Installed providers:", ort.get_available_providers())
print("Key: an installed provider is not evidence that the whole graph uses that accelerator.")


## 1. 固定 Eager 参考和输入契约

模型入口为 `[batch, tokens, 16]` 的 FP32 张量，batch 范围 1–8，tokens 范围 2–64；输出最后一维为 8。这个小模型只验证导出机制，不模拟完整 LLM 的 KV、采样或 tokenization。


In [ ]:
class TinyTokenProjector(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.up = torch.nn.Linear(16, 32)
        self.down = torch.nn.Linear(32, 8)

    def forward(self, x):
        return self.down(torch.nn.functional.silu(self.up(x)))


model = TinyTokenProjector().eval()
example = torch.randn(2, 8, 16)
with torch.inference_mode():
    baseline = model(example)
print("Reference output shape:", tuple(baseline.shape))
print("Key: fix weights, dtype, and shape constraints before comparing runtimes.")


## 2. 显式选择现代导出路径

`dynamo=True` 选择基于 torch.export 的导出器，`dynamic_shapes` 使用 Python 入口参数名 `x`。opset 18 是本实验选择，需与目标后端支持对齐。

这里用临时目录存放产物，最后清理。生产交付还需保存权重外部数据文件、tokenizer、模型配置等完整依赖；本模型较小，权重直接放在单个 ONNX 文件中。


In [ ]:
workspace = tempfile.TemporaryDirectory(prefix="edge_export_parity_")
artifact_path = Path(workspace.name) / "tiny_token_projector.onnx"
export_started = perf_counter()
try:
    torch.onnx.export(
        model,
        (example,),
        artifact_path,
        dynamo=True,
        input_names=["x"],
        output_names=["y"],
        dynamic_shapes={
            "x": {
                0: torch.export.Dim("batch", min=1, max=8),
                1: torch.export.Dim("tokens", min=2, max=64),
            }
        },
        opset_version=18,
        external_data=False,
    )
except Exception:
    workspace.cleanup()
    raise
export_ms = (perf_counter() - export_started) * 1000
onnx.checker.check_model(str(artifact_path), full_check=True)
print(f"Export and serialization: {export_ms:.2f} ms")
print("ONNX schema/operator checks passed.")
print("Key: a valid ONNX graph still needs runtime and numerical validation.")


## 3. 指定运行后端

显式使用 CPUExecutionProvider，报告实际 session provider 和输入形状。NPU / GPU 验证还需相应执行提供程序、算子分区、fallback 记录与目标设备测量；这个 CPU 会话不提供这些证据。


In [ ]:
options = ort.SessionOptions()
options.intra_op_num_threads = 1
options.inter_op_num_threads = 1
runtime_started = perf_counter()
session = ort.InferenceSession(
    str(artifact_path),
    sess_options=options,
    providers=["CPUExecutionProvider"],
)
runtime_init_ms = (perf_counter() - runtime_started) * 1000
print(f"Runtime initialization: {runtime_init_ms:.2f} ms")
input_info = session.get_inputs()[0]
assert input_info.name == "x"
assert session.get_providers() == ["CPUExecutionProvider"]
print("Session providers:", session.get_providers())
print("Input:", input_info.name, input_info.type, input_info.shape)
print("Output:", session.get_outputs()[0].name, session.get_outputs()[0].shape)
print("Key: runtime initialization is a separate gate from model export.")


## 4. 多形状回归，包含边界

从有效范围取最小、典型、最大和另一个 batch；分别比较完整输出，而不是只检查“能运行”或输出形状。容差依赖精度与算子；量化后还需要任务级评测，不能沿用 FP32 的结论。


In [ ]:
errors = []
with torch.inference_mode():
    for shape in ((1, 2, 16), (2, 8, 16), (8, 64, 16), (3, 17, 16)):
        tensor = torch.randn(*shape)
        reference = model(tensor).numpy()
        actual = session.run(["y"], {"x": tensor.numpy()})[0]
        np.testing.assert_allclose(actual, reference, rtol=1e-4, atol=1e-5)
        maximum_error = float(np.max(np.abs(actual - reference)))
        errors.append(maximum_error)
        print(f"Shape {shape}: max absolute error {maximum_error:.8f}")
print("Key: test shapes other than the example used during export.")


## 5. 在运行时入口明确执行范围约束

不要把 ONNX 的符号维度名视为完整保留并强制执行 torch.export 的所有上下界。调用方显式检查 rank、dtype、固定维度、batch/token 范围与有限数值，再交给 session。

下列验证器是本实验的交付契约；它不试图覆盖任意模型的输入规则。


In [ ]:
def validate_input(array):
    if not isinstance(array, np.ndarray) or array.ndim != 3:
        raise ValueError("Expected a rank-3 numpy array.")
    if array.dtype != np.float32:
        raise ValueError("Expected float32 input.")
    batch, tokens, hidden = array.shape
    if not (1 <= batch <= 8 and 2 <= tokens <= 64 and hidden == 16):
        raise ValueError("Input violates batch/token/hidden constraints.")
    if not np.isfinite(array).all():
        raise ValueError("Input must contain finite values.")


def run_checked(array):
    validate_input(array)
    return session.run(["y"], {"x": array})[0]


assert run_checked(np.zeros((1, 2, 16), dtype=np.float32)).shape == (1, 2, 8)
invalid_inputs = [
    np.zeros((9, 8, 16), dtype=np.float32),
    np.zeros((2, 65, 16), dtype=np.float32),
    np.zeros((2, 8, 15), dtype=np.float32),
    np.zeros((2, 8, 16), dtype=np.float64),
    np.full((2, 8, 16), np.nan, dtype=np.float32),
]
for array in invalid_inputs:
    try:
        run_checked(array)
    except ValueError:
        print("Rejected:", array.shape, array.dtype)
    else:
        raise AssertionError("Expected contract rejection.")
print("Key: input constraints are part of the deployed interface.")


## 6. 分开记录导出、初始化和稳态

下面只测 ONNX Runtime 的预热后前向，包含 Python 调用与运行时开销，不包含输入验证、模型加载或导出，不是端到端 LLM 的 TTFT / tokens/s。7 次试验各运行 20 次，报告每次平均耗时的中位数。


In [ ]:
timing_input = np.ones((2, 32, 16), dtype=np.float32)
for _ in range(3):
    session.run(["y"], {"x": timing_input})
trial_ms = []
for _ in range(7):
    started = perf_counter()
    for _ in range(20):
        session.run(["y"], {"x": timing_input})
    trial_ms.append((perf_counter() - started) * 1000 / 20)
warm_ms = statistics.median(trial_ms)
print(f"CPU runtime warm forward median: {warm_ms:.4f} ms")
print("Key: state the timing boundary and avoid comparing unlike workloads.")


## 7. 交付元信息与产物指纹

记录导出工具、运行时、opset、输入契约、验证结果及文件 hash。hash 用于识别内容，不能替代签名、授权或来源证明。OTA、模型安全与质量门禁可复用这些字段。


In [ ]:
loaded_graph = onnx.load(artifact_path)
artifact_bytes = artifact_path.read_bytes()
manifest = {
    "torch_version": torch.__version__,
    "onnx_version": onnx.__version__,
    "runtime_version": ort.__version__,
    "providers": session.get_providers(),
    "opsets": {entry.domain or "ai.onnx": entry.version for entry in loaded_graph.opset_import},
    "input_contract": {"dtype": "float32", "batch": [1, 8], "tokens": [2, 64], "hidden": 16},
    "max_absolute_error": max(errors),
    "artifact_bytes": len(artifact_bytes),
    "sha256": hashlib.sha256(artifact_bytes).hexdigest(),
    "export_ms": export_ms,
    "runtime_init_ms": runtime_init_ms,
    "warm_forward_ms": warm_ms,
    "timing_scope": "warm CPU forward only",
}
print(json.dumps(manifest, indent=2))
del session
workspace.cleanup()
print("Temporary ONNX artifact cleaned up.")
print("Key: carry compatibility and validation evidence with the model artifact.")


## 思考题

1. ONNX 校验通过、session 初始化成功、数值对齐，分别证明什么？
2. 为什么动态维度还需要调用侧范围验证？
3. 目标设备有 NPU，但一半算子在 CPU 上运行，会影响哪些测量？
4. 模型升级后，为什么还要重新检查 tokenizer、输入契约和精度？
5. 应如何把本实验的 manifest 接入模型 OTA 和质量门禁？

## 官方参考

- [PyTorch ONNX 导出器](https://docs.pytorch.org/docs/2.15/onnx.html)
- [ONNX Runtime Execution Providers](https://onnxruntime.ai/docs/execution-providers/)
- [ExecuTorch 1.6 发布记录](https://github.com/pytorch/executorch/releases/tag/v1.6.0)
- [课程模型 OTA 实验](../19_model_ota/19.1_model_ota.ipynb)
- [课程质量门禁实验](../24_edge_mlops/24.1_edge_mlops.ipynb)
